# Fetching job postings

In [1]:
import asyncio
from playwright.async_api import async_playwright

async def scrape_jobs():
    jobs_data = []
    category = "data-science-internship"  # Change this to your desired category

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        await page.goto(
            "https://internshala.com/internships/"+category, 
            wait_until="networkidle"
        )
        await page.wait_for_selector(".job-title-href")

        # ✅ Only real job cards (no ads)
        job_cards = page.locator(".internship_meta:has(.job-title-href)")
        count = await job_cards.count()

        print("Real jobs found:", count)

        for i in range(count):
            card = job_cards.nth(i)

            # Title
            title_locator = card.locator(".job-title-href").first
            title = (await title_locator.text_content()).strip()

            # Link
            relative_link = await title_locator.get_attribute("href")
            link = f"https://internshala.com{relative_link}"

            # Company
            company = (await card.locator(".company-name").text_content()).strip()

            # Location
            location = (await card.locator(".locations span").first.text_content()).strip()

            # Stipend
            stipend = (await card.locator(".stipend").text_content()).strip()

            # Duration (3 Months etc.)
            duration = (await card.locator(".row-1-item").nth(2).text_content()).strip()

            # Description
            description = (await card.locator(".about_job .text").text_content()).strip()

            # Days since posted
            posted_locator = card.locator(".status-inactive span")
            days_since_posted = (
                (await posted_locator.text_content()).strip()
                if await posted_locator.count() > 0
                else None
            )

            jobs_data.append({
                "title": title,
                "link": link,
                "company": company,
                "location": location,
                "stipend": stipend,
                "duration": duration,
                "description": description,
                "days_since_posted": days_since_posted
            })

        await browser.close()

    return jobs_data


jobs = await scrape_jobs()
jobs[:3]

Real jobs found: 50


[{'title': 'Executive Assistant (AI & MS Office)',
  'link': 'https://internshala.com/internship/detail/executive-assistant-ai-ms-office-internship-in-lucknow-at-zagrication-llp1770099753',
  'company': 'ZAgrication LLP',
  'location': 'Lucknow                                                                (Hybrid)',
  'stipend': '₹ 8,000 - 12,000 /month',
  'duration': '3 Months',
  'description': '1. Manage daily schedules, reminders, follow-ups, and task coordination.\n2. Create trackers, reports, documents, and presentations using MS Office.\n3. Use AI tools to draft emails, summaries, SOPs, and structured notes.\n4. Coordinate with internal teams and ensure timely task closure.\n5. Maintain organised records and documentation.',
  'days_since_posted': '3 weeks ago'},
 {'title': 'Performance Marketing',
  'link': 'https://internshala.com/internship/detail/performance-marketing-internship-in-mumbai-at-expanse-digital1769691420',
  'company': 'Expanse Digital',
  'location': 'Mumbai'

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent.parent))

from src.platforms.internshala import IntershalaPlatform
import asyncio
from playwright.async_api import async_playwright
from src.storage.resume_store import load_resume_summary
from src.storage.application_tracker import (
    load_preferences_md,
    load_raw_jobs
)


async def scrape_jobs():
    jobs_data = []
    category = "data-science-internship"  # Change this to your desired category

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="state.json")
        
        page = await context.new_page()
        # search_url = f"https://internshala.com/internships/{category}"
        # await page.goto(
        #     'https://internshala.com/internships/data-analyst-internship/', 
        #     wait_until="networkidle"
        # )
        # jobs= load_raw_jobs()
        resume_summary= load_resume_summary()
        preferences_md = load_preferences_md()
        platform = IntershalaPlatform()
        jobs = await platform.scrape_jobs(page,"https://internshala.com/internships/data-science-internship/")
        # result = await platform.apply(page,jobs[6],resume_summary,preferences_md)
        print(jobs)
        # print()
        
await scrape_jobs()

[03/05/26 07:09:09] INFO     Scraping page 1: https://internshala.com/internships/data-science-internship/

[03/05/26 07:09:26] INFO       Found 50 jobs on page 1

                    INFO     More than 40 jobs found in the first page itself.

                    INFO     Total scraped: 50 jobs

[{'title': 'Social Media Marketing', 'company': 'Scribbld Social', 'link': 'https://internshala.com/internship/detail/social-media-marketing-internship-in-multiple-locations-at-scribbld-social1772185958', 'location': 'Navi Mumbai, Mumbai', 'stipend': '₹ 5,000 - 6,000 /month', 'duration': '3 Months', 'days_posted': None, 'description': 'Develop engaging social media content, schedule posts, and monitor community interactions', 'experience': None, 'platform': 'internshala'}, {'title': 'Operations', 'company': 'Vijay Social Welfare Society', 'link': 'https://internshala.com/internship/detail/operations-internship-in-indore-at-vijay-social-welfare-society1772171991', 'location': 'Indore                                        (Hybrid)', 'stipend': 'Unpaid', 'duration': '2 Months', 'days_posted': None, 'description': 'Analyze data, manage events, and create social media awareness for child education and environmental initiatives', 'experience': None, 'platform': 'internshala'}, {'title': 'Le

'**Fahad Noufal**\nAI ML Engineer\nfhdnaufal@gmail.com | (+91) 8590988835\nBangalore\nPortfolio: fahadnoufal.github.io | Github: github.com/fahadnoufal\n\n**Key Skills:** Gen AI & LLM Ecosystem, RAG, Fine-tuning (PEFT/LoRA), Agentic AI, LangGraph, VectorDB, Gemini, OpenAI, Llama 3, Unsloth, Prompt Engineering, ML & DevOps, Pandas, PyTorch, Plotly, SQL, Docker, FastAPI, AWS, GCP, CI/CD, GitHub Actions, Python, JavaScript.\n\n**Education:**\n*   BSc. Data Science | Kristu Jayanti University | Bangalore, India | CGPA: 8.0 | 2025\n\n**Work/Internship Experience:**\n*   **Data Scientist Intern** | MainFlow Technologies | Bangalore | 2024\n    *   Built ML models using structured and unstructured data to improve data-driven workflows and business intelligence.\n    *   Collaborated with cross-functional teams to refine preprocessing pipelines and ensure high-quality data ingestion.\n*   **Web Developer (Community Lead)** | Google Developer Student Club | 2023 – 2024\n    *   Led the UI/UX de

In [ ]:
from src.llm.generator import generate_answers
platform = IntershalaPlatform()


# Authenticating to internshala

In [ ]:
import time
import os
async def login():
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://internshala.com")
        print("Login manually in this window...")

        input("Press ENTER after login is complete...")

        await context.storage_state(path="internshala_state.json")
        
apply = await login()

Login manually in this window...


# Applying to the job

In [98]:
from __future__ import annotations

import asyncio
import json
import logging
from dataclasses import dataclass, field, asdict
from typing import Any

from playwright.async_api import Page, async_playwright

log = logging.getLogger(__name__)

@dataclass
class QuestionDict:
    heading: str
    type: str                       # text | radio | checkbox | select | file | availability
    question: str
    options: list[str]
    description: str | None
    required: bool
    field_name: str | None = None   # HTML name attribute
    field_id: str | None = None     # HTML id attribute
    conditional: bool = False     

In [111]:
async def extract_form_questions(page: Page) -> list[QuestionDict]:
    """
    Dynamically scrape every question inside #assessment_questions.
    Returns a list of QuestionDict (also serialisable via dataclasses.asdict).
    """
    raw: list[dict] = await page.evaluate(_EXTRACTOR_JS)
    return [QuestionDict(**q) for q in raw]


# JavaScript run inside the browser — returns plain JSON-serialisable dicts
_EXTRACTOR_JS = """
() => {
  const questions = [];

  // ── tiny helpers ──────────────────────────────────────────────────────────

  const clean = s => (s || '').replace(/\\s+/g, ' ').trim();

  function hasRequired(container) {
    if (container.querySelector("[aria-required='true']")) return true;
    // Internshala adds .form-error labels for required fields
    const err = container.querySelector(".form-error");
    if (err) return true;
    return false;
  }

  function radioOptions(container) {
    return Array.from(container.querySelectorAll("input[type='radio']")).map(r => {
      const lbl = container.querySelector(`label[for='${r.id}']`);
      return clean(lbl ? lbl.textContent : r.value);
    });
  }

  function checkboxOptions(container) {
    return Array.from(container.querySelectorAll("input[type='checkbox']")).map(c => {
      const lbl = container.querySelector(`label[for='${c.id}']`);
      return clean(lbl ? lbl.textContent : c.value);
    });
  }

  // ── root ──────────────────────────────────────────────────────────────────

  const root = document.querySelector("#assessment_questions");
  if (!root) return questions;

  // ── 1. Cover Letter ───────────────────────────────────────────────────────
  const coverGroup = root.querySelector(".form-group:has(#cover_letter_holder), .form-group:has(#cover_letter)");
  if (coverGroup) {
    const headingEl = root.querySelector("h4.question-heading");
    const labelEl   = coverGroup.querySelector(".assessment_question label");
    questions.push({
      heading:     clean(headingEl ? headingEl.textContent : "Cover Letter"),
      type:        "text",
      question:    clean(labelEl ? labelEl.textContent : "Why should you be hired for this role?"),
      options:     [],
      description: null,
      required:    hasRequired(coverGroup),
      field_name:  "cover_letter",
      field_id:    "cover_letter",
      conditional: false,
    });
  }

  // ── 2. Confirm Availability ───────────────────────────────────────────────
  const availSection = root.querySelector("#confirm_availability_container");
  if (availSection) {
    const heading = clean(availSection.querySelector(".question-heading")?.textContent || "Confirm your availability");

    questions.push({
      heading,
      type:        "availability",    // specialised radio with conditional sub-fields
      question:    heading,
      options:     radioOptions(availSection),
      description: null,
      required:    true,
      field_name:  "confirm_availability",
      field_id:    null,
      conditional: false,
    });
  }

  // ── 3. Additional / Custom Questions ─────────────────────────────────────
  const addContainer = root.querySelector(".questions-container");
  if (addContainer) {
    const sectionHeading = clean(
      addContainer.querySelector(".additional-question-heading, h4")?.textContent
      || "Additional questions"
    );

    addContainer.querySelectorAll(".additional_question, .form-group.additional_question").forEach(block => {
      const labelEl = block.querySelector(".assessment_question label");
      const question = clean(labelEl ? labelEl.textContent : "");
      if (!question) return;

      // Radio
      const radios = block.querySelectorAll("input[type='radio']");
      if (radios.length) {
        questions.push({
          heading:     sectionHeading,
          type:        "radio",
          question,
          options:     radioOptions(block),
          description: null,
          required:    hasRequired(block),
          field_name:  radios[0].name,
          field_id:    null,
          conditional: false,
        });
        return;
      }

      // Checkbox
      const checks = block.querySelectorAll("input[type='checkbox']");
      if (checks.length) {
        questions.push({
          heading:     sectionHeading,
          type:        "checkbox",
          question,
          options:     checkboxOptions(block),
          description: null,
          required:    hasRequired(block),
          field_name:  checks[0].name,
          field_id:    null,
          conditional: false,
        });
        return;
      }

      // Select / dropdown
      const sel = block.querySelector("select");
      if (sel) {
        const opts = Array.from(sel.options).filter(o => !o.disabled).map(o => clean(o.textContent));
        questions.push({
          heading:     sectionHeading,
          type:        "select",
          question,
          options:     opts,
          description: null,
          required:    hasRequired(block),
          field_name:  sel.name,
          field_id:    sel.id || null,
          conditional: false,
        });
        return;
      }

      // Visible textarea
      const ta = block.querySelector("textarea:not([style*='display: none'])");
      if (ta) {
        questions.push({
          heading:     sectionHeading,
          type:        "text",
          question,
          options:     [],
          description: clean(ta.getAttribute("placeholder") || ""),
          required:    hasRequired(block),
          field_name:  ta.name || null,
          field_id:    ta.id   || null,
          conditional: false,
        });
        return;
      }

      // Text / email / number input
      const inp = block.querySelector("input[type='text'],input[type='email'],input[type='number']");
      if (inp) {
        questions.push({
          heading:     sectionHeading,
          type:        "text",
          question,
          options:     [],
          description: clean(inp.getAttribute("placeholder") || ""),
          required:    hasRequired(block),
          field_name:  inp.name || null,
          field_id:    inp.id   || null,
          conditional: false,
        });
        return;
      }

      // File upload inside additional questions
      const fileInp = block.querySelector("input[type='file']");
      if (fileInp) {
        questions.push({
          heading:     sectionHeading,
          type:        "file",
          question,
          options:     [],
          description: null,
          required:    hasRequired(block),
          field_name:  fileInp.name || null,
          field_id:    fileInp.id   || null,
          conditional: false,
        });
      }
    });
  }

  return questions;
}
"""

In [ ]:

async def fill_form_answers(
    page: Page,
    questions: list[QuestionDict],
    answers: list[Any],
) -> None:
    """
    Fill the Internshala application form.

    Parameters
    ----------
    page      : Playwright page (form already visible)
    questions : output of extract_form_questions()
    answers   : list aligned 1-to-1 with questions
                  text        → str
                  radio       → str  (option label)
                  availability→ str  (option label)
                  checkbox    → list[str] | str
                  select      → str  (option label)
                  file        → str  (absolute path)  or None to skip
    """
    if len(answers) != len(questions):
        raise ValueError(
            f"answers length ({len(answers)}) != questions length ({len(questions)})"
        )

    for q, answer in zip(questions, answers):
        if answer is None:
            log.debug("Skipping %r (answer is None)", q.question)
            continue

        try:
            match q.type:
                case "text":
                    await _fill_text(page, q, answer)
                case "radio" | "availability":
                    await _fill_radio(page, q, answer)
                case "checkbox":
                    await _fill_checkbox(page, q, answer)
                case "select":
                    await _fill_select(page, q, answer)
                case _:
                    log.warning("Unknown question type %r — skipped", q.type)

        except Exception as exc:  # noqa: BLE001
            log.warning("Could not fill %r: %s", q.question, exc)


# ── per-type helpers ───────────────────────────────────────────────────────────

async def _fill_text(page: Page, q: QuestionDict, answer: str) -> None:
    """Fill textarea / input, handling Quill rich-text editors."""
    answer = str(answer)

    # Cover letter uses a Quill editor — the real <textarea> is hidden
    if q.field_id == "cover_letter":
        editor = page.locator(".ql-editor").first
        await editor.click()
        # Select all existing text and replace
        await editor.press("Control+a")
        await editor.type(answer, delay=1)
        return

    # For all other text fields, use the field_id or field_name to target specifically
    if q.field_id:
        # Try by ID first (most specific)
        selector = f"#{q.field_id}"
        el = page.locator(selector).first
        
        if await el.count() == 0 and q.field_name:
            # Fallback to name attribute
            selector = f"textarea[name='{q.field_name}'], input[name='{q.field_name}']"
            el = page.locator(selector).first
    elif q.field_name:
        # Only have field_name
        selector = f"textarea[name='{q.field_name}'], input[name='{q.field_name}']"
        el = page.locator(selector).first
    else:
        raise ValueError(f"Question must have either field_id or field_name: {q}")

    # Fill the field
    await el.click()
    await el.fill(answer)


async def _fill_radio(page: Page, q: QuestionDict, answer: str) -> None:
    """
    Click the radio whose label text matches `answer` (case-insensitive,
    substring OK).  Falls back to matching the value attribute.
    """
    answer_lower = answer.strip().lower()

    clicked = await page.evaluate(
        """([field_name, answer_lower]) => {
            const radios = document.querySelectorAll(`input[type='radio'][name='${field_name}']`);
            for (const r of radios) {
                const lbl = document.querySelector(`label[for='${r.id}']`);
                const text = (lbl ? lbl.textContent : r.value).toLowerCase().trim();
                if (text.includes(answer_lower) || r.value.toLowerCase() === answer_lower) {
                    r.click();
                    return true;
                }
            }
            return false;
        }""",
        [q.field_name, answer_lower],
    )

    if not clicked:
        log.warning(
            "Radio option %r not found for question %r (options: %s)",
            answer, q.question, q.options,
        )


async def _fill_checkbox(page: Page, q: QuestionDict, answer: list[str] | str) -> None:
    """
    Check boxes whose labels are in `answer` (list); uncheck the rest.
    `answer` may be a single string (treated as a one-element list).
    """
    targets = [a.strip().lower() for a in (answer if isinstance(answer, list) else [answer])]

    await page.evaluate(
        """([field_name, targets]) => {
            const boxes = document.querySelectorAll(`input[type='checkbox'][name='${field_name}']`);
            for (const cb of boxes) {
                const lbl = document.querySelector(`label[for='${cb.id}']`);
                const text = (lbl ? lbl.textContent : cb.value).toLowerCase().trim();
                const should = targets.some(t => text.includes(t) || cb.value.toLowerCase() === t);
                if (should !== cb.checked) cb.click();
            }
        }""",
        [q.field_name, targets],
    )


async def _fill_select(page: Page, q: QuestionDict, answer: str) -> None:
    """
    Select a <select> option by label (case-insensitive substring match).
    Uses selectOption with value as fallback.
    """
    answer_lower = answer.strip().lower()

    # Resolve exact option value from label text
    option_value: str | None = await page.evaluate(
        """([field_name, answer_lower]) => {
            const sel = document.querySelector(`select[name='${field_name}']`);
            if (!sel) return null;
            for (const opt of sel.options) {
                if (opt.disabled) continue;
                if (opt.textContent.toLowerCase().trim().includes(answer_lower)) return opt.value;
            }
            return null;
        }""",
        [q.field_name, answer_lower],
    )

    selector = f"select[name='{q.field_name}']"
    if option_value is not None:
        await page.select_option(selector, value=option_value)
    else:
        # Last-resort: try passing the answer as a label directly
        await page.select_option(selector, label=answer)



# ─── Convenience wrapper ──────────────────────────────────────────────────────

async def process_application(
    page: Page,
    llm_answers: list[Any],
) -> list[QuestionDict]:
    """
    High-level helper:
      1. Extract questions from the currently-open form
      2. Print them (so you can pass to your LLM)
      3. Fill with the provided answers

    Returns the extracted questions for inspection / logging.
    """
    questions = await extract_form_questions(page)
    log.info("Extracted %d questions", len(questions))

    # Pretty-print for debugging / LLM prompt building
    print("\n── Extracted Questions ──────────────────────────────────────")
    print(json.dumps([asdict(q) for q in questions], indent=2))
    print("─────────────────────────────────────────────────────────────\n")

    await fill_form_answers(page, questions, llm_answers)
    return questions


In [ ]:

from playwright.async_api import async_playwright
import asyncio
import json

fake_answers = [
        # Cover letter
        (
            "I am a passionate AI engineer with 2 years of hands-on experience building "
            "LLM-powered agents using LangChain and OpenAI APIs. I have shipped a production "
            "RAG system for a legal-tech startup and I am eager to bring that expertise to "
        ),
        "Yes, I am available to join immediately",
        'Yes'
    ]

async def apply_job():
    async with async_playwright() as p:
        global questions
        browser = await p.chromium.launch(headless=False)

        # ✅ Load saved login session
        context = await browser.new_context(storage_state="internshala_state.json")

        page = await context.new_page()
        
        await page.goto(current_job['link'], wait_until="networkidle")
        
        # clicking apply
        await page.click("#top_easy_apply_button")
        
        # getting questions
        await page.wait_for_selector("#assessment_questions")
        

        # questions = await extract_form_questions(page)
        
        questions = await process_application(page, fake_answers)

        print("Form filled successfully ✓")
        print(f"Total questions handled: {len(questions)}")
        
        await page.click("#submit")
        
        
        input('enter conform')
                
        # input('enter when completed:')

apply = await apply_job()

INFO  Extracted 3 questions



── Extracted Questions ──────────────────────────────────────
[
  {
    "heading": "Cover letter",
    "type": "text",
    "question": "Why should you be hired for this role?",
    "options": [],
    "description": null,
    "required": true,
    "field_name": "cover_letter",
    "field_id": "cover_letter",
    "conditional": false
  },
  {
    "heading": "Confirm your availability",
    "type": "availability",
    "question": "Confirm your availability",
    "options": [
      "Yes, I am available to join immediately",
      "No, I am currently on notice period",
      "No, I will have to serve notice period",
      "Other (Please specify your availability)"
    ],
    "description": null,
    "required": true,
    "field_name": "confirm_availability",
    "field_id": null,
    "conditional": false
  },
  {
    "heading": "Additional question(s)",
    "type": "radio",
    "question": "Do you have a working laptop and internet?",
    "options": [
      "Yes",
      "No"
    ],
    "de

In [119]:
questions

[QuestionDict(heading='Confirm your availability', type='availability', question='Confirm your availability', options=['Yes, I am available to join immediately and can work from 8:30 pm - 5:30 am Indian Standard Time (9:00 am - 6:00 pm Central Standard Time)', 'No, I am currently on notice period', 'No, I will have to serve notice period', 'Other (Please specify your availability)'], description=None, required=True, field_name='confirm_availability', field_id=None, conditional=False),
 QuestionDict(heading='Additional question(s)', type='text', question="Drop your GitHub link. What's the most impressive thing you've built? (2-3 sentences - sell us on it).", options=[], description='Enter text ...', required=True, field_name='custom_question_text_6974805', field_id='custom_question_text_6974805', conditional=False),
 QuestionDict(heading='Additional question(s)', type='text', question="You're handed a 2,000-word build spec for a Vertex AI Search pipeline and told to ship it by Friday wi